# 09 - TabKAN: ChebyshevKAN no churn de telecom

Quanto muda o KS de validação ao usar polinômios de Chebyshev nas conexões? A comparação emparelha cada fold com a KAN de três intervalos da E0. O fold 1 orienta a escolha e os outros dois descrevem a consistência do resultado.

Esta implementação local em PyTorch usa a variante polinomial do [TabKAN](https://github.com/aseslamian/TabKAN) com o treinamento compartilhado do projeto.

In [1]:
# Localiza o projeto a partir da raiz ou da pasta de notebooks.
from pathlib import Path
import sys

raiz = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/config.py").exists())
sys.path.insert(0, str(raiz))
import pandas as pd
import torch
from src import config as cfg
from src.kan import ajustar, criar_spline
from src.tabkan import criar_chebyshev
from src.preprocess import load_preprocessed
from src.metrics import evaluate_sets, save_results

torch.set_num_threads(1)

## E0 - Base de Chebyshev de grau 3

Cada conexão recebe uma entrada escalar comprimida por tanh e aprende uma soma ponderada dos termos de graus 0 a 3. Isso permite curvatura suave ao longo da entrada. A transformação da entrada e a quantidade de coeficientes também diferem entre as arquiteturas.

Mantemos a variante neural, a largura 10, o Adam, a semente 42 e a regra de parada da KAN. A perda de validação seleciona a época. O KS mede a separação entre os escores das duas classes. O teste fica reservado à avaliação final.

In [2]:
# Treina apenas com as partições destinadas ao desenvolvimento.
config = dict(largura=10, grau=3, lr=0.001, lote=64, max_epocas=120,
              paciencia=15, min_delta=0.0001)
linhas = []
for fold in range(1, cfg.K_FOLDS + 1):
    dados = {c: load_preprocessed(fold, c, "neural") for c in ("treino", "validacao")}
    entradas = dados["treino"][0].shape[1]
    cheb = criar_chebyshev(entradas, config)
    spline = criar_spline(entradas, dict(largura=10, intervalos=3))
    n_cheb = sum(p.numel() for p in cheb.parameters())
    n_spline = sum(p.numel() for p in spline.parameters())
    scores, historia, epoca, segundos = ajustar(dados, config, cfg.SEED, criar_chebyshev)
    parametros = dict(config, parametros_treinaveis=n_cheb, melhor_epoca=epoca, epocas_rodadas=len(historia))
    linhas.append(evaluate_sets("tabkan", "neural", fold, scores, hiperparametros=parametros,
                                etapa="E0_baseline", semente=cfg.SEED))
    print(f"Fold {fold}: {n_cheb} parâmetros (KAN: {n_spline}), época {epoca}/{len(historia)}, {segundos:.1f} s")
resultado = pd.concat(linhas, ignore_index=True)
save_results(resultado)

Fold 1: 1640 parâmetros (KAN: 1241), época 7/22, 2.2 s
Fold 2: 1640 parâmetros (KAN: 1241), época 5/20, 0.5 s
Fold 3: 1640 parâmetros (KAN: 1241), época 11/26, 0.7 s


In [3]:
# Calcula a diferença de KS para a mesma etapa, semente, variante e fold.
kan = pd.read_csv(cfg.REPORTS_TABLES / "resultados_kan.csv")
comparacao = pd.concat([kan, resultado]).query(
    "etapa == 'E0_baseline' and semente == @cfg.SEED and variante == 'neural' and conjunto == 'validacao'"
).pivot(index="fold", columns="modelo", values="ks")
comparacao["delta"] = comparacao["tabkan"] - comparacao["kan"]
comparacao.loc["Média"] = comparacao.mean()
print(comparacao.round(4).to_string())

modelo     kan  tabkan   delta
fold                          
1       0.5043  0.5078  0.0035
2       0.5157  0.5302  0.0145
3       0.5386  0.5386 -0.0000
Média   0.5195  0.5255  0.0060


## Registro do processo

O KS médio da ChebyshevKAN foi 0.5255, frente a 0.5195 da KAN. No fold 1, a diferença foi +0.0035. A ChebyshevKAN superou a referência em 2 dos três folds. Esta rodada cobre uma única semente. Uma próxima rodada pode reduzir o grau para 2 e verificar se o termo cúbico contribui para o KS.